# Extended Thinking con Claude Opus 4.7

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-4/02-extended-thinking-opus.ipynb)

Extended Thinking permite a Claude Opus 4.7 razonar en profundidad antes de responder. En este notebook veremos cómo activarlo y cuándo usarlo.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
client = anthropic.Anthropic()

## 1. Ejemplo básico con Extended Thinking

In [ ]:
response = client.beta.messages.create(
    model='claude-opus-4-7',
    max_tokens=16000,
    thinking={
        'type': 'enabled',
        'budget_tokens': 8000,
    },
    betas=['interleaved-thinking-2025-05-14'],
    messages=[{
        'role': 'user',
        'content': 'Si lanzo dos dados de 6 caras, ¿cuál es la probabilidad de que la suma sea mayor que 9? Muestra el razonamiento completo.'
    }],
)

for block in response.content:
    if block.type == 'thinking':
        print('=== RAZONAMIENTO INTERNO (primeros 500 chars) ===')
        print(block.thinking[:500])
        print('...')
        print(f'[Total thinking: {len(block.thinking)} caracteres]\n')
    elif block.type == 'text':
        print('=== RESPUESTA FINAL ===')
        print(block.text)

## 2. Comparativa: con y sin Extended Thinking

Veamos la diferencia de calidad en un problema de lógica.

In [ ]:
problema_logica = """Ana es más alta que Beatriz. Carla es más baja que Beatriz pero más alta que Diana.
Eva es más alta que Ana. ¿Cuál es el orden de mayor a menor altura?"""

# Sin thinking
r_sin = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=256,
    messages=[{'role': 'user', 'content': problema_logica}],
)

# Con thinking (Opus)
r_con = client.beta.messages.create(
    model='claude-opus-4-7',
    max_tokens=8000,
    thinking={'type': 'enabled', 'budget_tokens': 4000},
    betas=['interleaved-thinking-2025-05-14'],
    messages=[{'role': 'user', 'content': problema_logica}],
)

print('SIN Extended Thinking (Sonnet):')
print(r_sin.content[0].text)

print('\nCON Extended Thinking (Opus):')
texto_con = next(b.text for b in r_con.content if b.type == 'text')
print(texto_con)

## 3. Estimación de coste con thinking

In [ ]:
def estimar_coste_opus(input_tokens: int, output_tokens: int, thinking_tokens: int) -> float:
    precio_input = 15.0 / 1_000_000
    precio_output = 75.0 / 1_000_000
    return input_tokens * precio_input + (output_tokens + thinking_tokens) * precio_output

# Escenarios
escenarios = [
    ('Sin thinking (Sonnet)',  500,  300,    0, 3.0,  15.0),
    ('Thinking 4K (Opus)',     500,  300, 4000, 15.0, 75.0),
    ('Thinking 16K (Opus)',    500,  800, 16000, 15.0, 75.0),
]

print(f"{'Escenario':<30} {'Coste estimado'}")
print('-' * 50)
for nombre, inp, out, think, p_in, p_out in escenarios:
    coste = inp * (p_in/1e6) + (out + think) * (p_out/1e6)
    print(f"{nombre:<30} ${coste:.4f} por llamada")

## Cuándo usar Extended Thinking

| Caso de uso | ¿Usar thinking? | Budget recomendado |
|-------------|-----------------|--------------------|
| Clasificación simple | ❌ No | — |
| Resumen de texto | ❌ No | — |
| Problema matemático multi-paso | ✅ Sí | 4000-8000 |
| Análisis legal complejo | ✅ Sí | 8000-16000 |
| Arquitectura de software | ✅ Sí | 8000-16000 |
| Lógica con múltiples restricciones | ✅ Sí | 4000-12000 |